In [2]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 320
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
model.print_trainable_parameters()

Unsloth 2026.9.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [5]:
from google.colab import files
uploaded = files.upload()  # select v3_final_train.jsonl

from datasets import load_dataset
dataset = load_dataset("json", data_files="v3_final_train.jsonl", split="train")
print(dataset)
print(dataset[0])

Saving v3_final_train.jsonl to v3_final_train.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 268
})
{'system': 'You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale.', 'user': 'MRK: price $157.43, RSI 64, company announced a new $10B share buyback program, 5d momentum -1.5%.', 'assistant': '{"action": "buy", "confidence": 0.66, "rationale": "A large buyback authorization signals management confidence in the stock\'s valuation and provides a technical demand floor going forward."}'}


In [6]:
def format_example(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user", "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = dataset.map(format_example)
print(dataset[0]["text"])

lengths = [len(tokenizer(ex["text"])["input_ids"]) for ex in dataset]
print("Token lengths — min:", min(lengths), "max:", max(lengths), "avg:", sum(lengths)/len(lengths))

Map:   0%|          | 0/268 [00:00<?, ? examples/s]

<|im_start|>system
You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale.<|im_end|>
<|im_start|>user
MRK: price $157.43, RSI 64, company announced a new $10B share buyback program, 5d momentum -1.5%.<|im_end|>
<|im_start|>assistant
{"action": "buy", "confidence": 0.66, "rationale": "A large buyback authorization signals management confidence in the stock's valuation and provides a technical demand floor going forward."}<|im_end|>

Token lengths — min: 109 max: 160 avg: 135.41417910447763


In [7]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/268 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/268 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 268 | Num Epochs = 2 | Total steps = 68
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.454379
2,2.456098
3,2.414587
4,2.188539
5,2.151123
6,1.984344
7,2.016484
8,1.695157
9,1.695681
10,1.672344


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-68/tokenizer_config.json.


In [8]:
model.save_pretrained("qwen_trading_lora_v3")
tokenizer.save_pretrained("qwen_trading_lora_v3")

!zip -r qwen_trading_lora_v3.zip qwen_trading_lora_v3
files.download("qwen_trading_lora_v3.zip")

Unsloth: Restored added_tokens_decoder metadata in qwen_trading_lora_v3/tokenizer_config.json.


  adding: qwen_trading_lora_v3/ (stored 0%)
  adding: qwen_trading_lora_v3/README.md (deflated 65%)
  adding: qwen_trading_lora_v3/adapter_config.json (deflated 59%)
  adding: qwen_trading_lora_v3/adapter_model.safetensors (deflated 8%)
  adding: qwen_trading_lora_v3/chat_template.jinja (deflated 71%)
  adding: qwen_trading_lora_v3/tokenizer.json (deflated 81%)
  adding: qwen_trading_lora_v3/tokenizer_config.json (deflated 89%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import warnings
warnings.filterwarnings("ignore")

FastLanguageModel.for_inference(model)
test_prompt = [
    {"role": "system", "content": "You are a trading signal generator. Given market data, output a JSON decision with action, confidence, and rationale."},
    {"role": "user", "content": "COIN: price $210.40, RSI 74, 5d momentum +11.8%, price up sharply after Bitcoin rallied 9% this week, volume 2x average."},
]
inputs = tokenizer.apply_chat_template(test_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=False)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"action": "sell", "confidence": 0.63, "rationale": "RSI of 74 combined with a 11.8% five-day move driven by an external crypto rally rather than company-specific fundamentals suggests the move is borrowed momentum. High odds of reversion once the external driver fades."}


In [10]:
import json, re

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

def run_eval(m, tok, eval_data, label=""):
    FastLanguageModel.for_inference(m)
    results = []
    for ex in eval_data:
        messages = [
            {"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
        ]
        inputs = tok.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
        outputs = m.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=False)
        raw = tok.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        parsed = extract_json(raw)
        true_label = json.loads(ex["assistant"])
        results.append({
            "user": ex["user"], "raw_output": raw, "parsed": parsed,
            "valid_json": parsed is not None,
            "true_action": true_label["action"],
            "pred_action": parsed["action"] if parsed else None,
            "correct": parsed is not None and parsed.get("action") == true_label["action"],
            "pred_confidence": parsed.get("confidence") if parsed else None,
        })
    return results

def summarize(results, label=""):
    n = len(results)
    valid = [r for r in results if r["valid_json"]]
    n_valid = len(valid)
    correct = [r for r in valid if r["correct"]]
    accuracy = len(correct) / n_valid if n_valid else 0
    print(f"=== {label} ===")
    print(f"Total examples: {n}")
    print(f"Valid JSON: {n_valid}/{n} ({100*n_valid/n:.1f}%)")
    print(f"Accuracy (on valid outputs): {len(correct)}/{n_valid} ({100*accuracy:.1f}%)")
    print("\nPer-class breakdown:")
    for action in ["buy", "sell", "hold"]:
        class_examples = [r for r in valid if r["true_action"] == action]
        class_correct = [r for r in class_examples if r["correct"]]
        n_class = len(class_examples)
        acc = len(class_correct)/n_class if n_class else 0
        pred_dist = {}
        for r in class_examples:
            pred_dist[r["pred_action"]] = pred_dist.get(r["pred_action"], 0) + 1
        print(f"  {action:5s}: {len(class_correct)}/{n_class} correct ({100*acc:.1f}%)  |  predicted as: {pred_dist}")
    print("\nConfidence calibration:")
    buckets = {"low (0-0.5)": [], "mid (0.5-0.7)": [], "high (0.7-1.0)": []}
    for r in valid:
        c = r["pred_confidence"] or 0
        if c < 0.5: buckets["low (0-0.5)"].append(r)
        elif c < 0.7: buckets["mid (0.5-0.7)"].append(r)
        else: buckets["high (0.7-1.0)"].append(r)
    for name, bucket in buckets.items():
        if bucket:
            acc = sum(r["correct"] for r in bucket)/len(bucket)
            print(f"  {name}: n={len(bucket)}, accuracy={100*acc:.1f}%")
    return {"n": n, "valid_pct": 100*n_valid/n, "accuracy": 100*accuracy}

In [11]:
import os
print("v3_final_eval.jsonl exists:", os.path.exists("v3_final_eval.jsonl"))
print("ood_eval.jsonl exists:", os.path.exists("ood_eval.jsonl"))

v3_final_eval.jsonl exists: False
ood_eval.jsonl exists: False


In [12]:
uploaded = files.upload()  # select BOTH v3_final_eval.jsonl and ood_eval.jsonl in this one dialog

Saving v3_final_eval.jsonl to v3_final_eval.jsonl
Saving ood_eval.jsonl to ood_eval.jsonl


In [13]:
uploaded = files.upload()  # select v_final_eval.jsonl AND ood_eval.jsonl together

eval_data = load_jsonl("v3_final_eval.jsonl")
ood_data = load_jsonl("ood_eval.jsonl")

ft_v3_templated = run_eval(model, tokenizer, eval_data, label="v3-templated")
summarize(ft_v3_templated, label="Fine-tuned v3 — templated eval")

ft_v3_ood = run_eval(model, tokenizer, ood_data, label="v3-ood")
summarize(ft_v3_ood, label="Fine-tuned v3 — OOD eval")

Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saving v3_final_eval.jsonl to v3_final_eval (1).jsonl
Saving ood_eval.jsonl to ood_eval (1).jsonl


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

=== Fine-tuned v3 — templated eval ===
Total examples: 88
Valid JSON: 88/88 (100.0%)
Accuracy (on valid outputs): 83/88 (94.3%)

Per-class breakdown:
  buy  : 28/29 correct (96.6%)  |  predicted as: {'buy': 28, 'hold': 1}
  sell : 27/28 correct (96.4%)  |  predicted as: {'sell': 27, 'buy': 1}
  hold : 28/31 correct (90.3%)  |  predicted as: {'buy': 1, 'hold': 28, 'sell': 2}

Confidence calibration:
  low (0-0.5): n=33, accuracy=93.9%
  mid (0.5-0.7): n=25, accuracy=88.0%
  high (0.7-1.0): n=30, accuracy=100.0%


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

=== Fine-tuned v3 — OOD eval ===
Total examples: 10
Valid JSON: 10/10 (100.0%)
Accuracy (on valid outputs): 6/10 (60.0%)

Per-class breakdown:
  buy  : 2/3 correct (66.7%)  |  predicted as: {'hold': 1, 'buy': 2}
  sell : 1/2 correct (50.0%)  |  predicted as: {'sell': 1, 'buy': 1}
  hold : 3/5 correct (60.0%)  |  predicted as: {'hold': 3, 'sell': 2}

Confidence calibration:
  low (0-0.5): n=4, accuracy=75.0%
  mid (0.5-0.7): n=5, accuracy=40.0%
  high (0.7-1.0): n=1, accuracy=100.0%


{'n': 10, 'valid_pct': 100.0, 'accuracy': 60.0}